# 🧪 Testing & Evaluation Patterns for LLM Applications

## Learning Objectives
In this notebook, you will learn:
1. **Unit Testing with Mocks** - Test LLM-calling code deterministically by mocking the LLM, without hitting a real API
2. **Integration Testing** - Validate real end-to-end LLM behavior against a small set of live test cases
3. **LLM-as-Judge Evaluation** - Score responses on correctness, relevance, clarity, and completeness using another LLM as grader
4. **Regression Testing** - Catch quality drift by re-scoring a chain against a fixed test suite over time
5. **LangSmith Evaluation Datasets** - Build persistent, versioned test suites and compare experiments (prompt v1 vs v2) in a production-style workflow

## Prerequisites
- Familiarity with LangChain LCEL (prompts, chains, `.invoke()`)
- `OPENAI_API_KEY` in your `.env` (used by the LLM calls throughout)
- `LANGCHAIN_API_KEY` / LangSmith account for the tracing and evaluation-dataset sections
- `pytest` and `unittest.mock` for the unit-testing section

> Converted from `04_testing_patterns.py` - part of **05 Production and Operations**.

---
## 🔧 Setup

This notebook mixes three kinds of tests you'll actually use in production: fast **unit tests** that mock the LLM, **integration tests** that hit a real model, and **LLM-as-judge evaluations** that score subjective quality. We start by importing everything needed across all sections and loading environment variables from `.env`.

In [ ]:
# ============================================================================
# SETUP: Imports and Environment
# ============================================================================
from unittest.mock import Mock, patch
from typing import Callable

import pytest
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
from langsmith import traceable, Client

from dotenv import load_dotenv

load_dotenv()

print("✅ Environment loaded and imports ready!")

---
## 🧪 Part 1: Unit Testing with Mocks

Unit tests should be fast, deterministic, and free of network calls. The trick for LLM-calling code is to **inject a mock LLM** so the test exercises your chain's logic (prompt formatting, response handling) without ever calling OpenAI.

### Key Concepts:
- **Dependency injection**: `QAChain` accepts an optional `llm` argument so a test can pass in a `Mock()` instead of a real client
- **`Mock().invoke.return_value`**: Configures what the fake LLM returns, so assertions are deterministic

### `QAChain`

A minimal Q&A chain: format a prompt, call the LLM, return the text response. It exists here purely as the system under test for the mock-based unit tests below.

In [ ]:
# ============================================================================
# QACHAIN: System Under Test
# ============================================================================
class QAChain:
    """Simple Q&A chain for testing."""

    def __init__(self, llm=None):
        self.llm = llm or ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.prompt = ChatPromptTemplate.from_template(
            "Answer this question: {question}"
        )

    def ask(self, question: str) -> str:
        prompt_value = self.prompt.invoke({"question": question})
        response = self.llm.invoke(prompt_value)
        return response.content

### ✅ Test: Mocked Response

Passes a `Mock()` as the LLM so the test never leaves the process. We assert both on the returned value and on the fact that `invoke` was called exactly once.

In [ ]:
# ============================================================================
# TEST: QA Chain with a Mocked LLM
# ============================================================================
def test_qa_chain_with_mock():
    """Test QA chain with mocked LLM."""
    # Create mock LLM
    mock_llm = Mock()
    mock_llm.invoke.return_value = AIMessage(content="Paris")

    # Test with mock
    chain = QAChain(llm=mock_llm)
    result = chain.ask("What is the capital of France?")

    assert result == "Paris"
    mock_llm.invoke.assert_called_once()

### ⚠️ Test: Empty Response Handling

Edge cases matter as much as happy paths — this confirms the chain doesn't choke when the LLM returns an empty string.

In [ ]:
# ============================================================================
# TEST: QA Chain Handles an Empty Response
# ============================================================================
def test_qa_chain_handles_empty_response():
    """Test chain handles empty responses."""
    mock_llm = Mock()
    mock_llm.invoke.return_value = AIMessage(content="")

    chain = QAChain(llm=mock_llm)
    result = chain.ask("Empty question")

    assert result == ""

---
## 🌐 Part 2: Integration Testing with Real LLM Calls

Unit tests with mocks tell you your code *plumbing* works. Integration tests tell you the **real model** actually produces the outputs you expect. These are slower and cost tokens, so keep the test-case count small and focused on core behaviors.

### `IntegrationTestSuite`

Runs a handful of real questions through `ChatOpenAI` and checks that the response contains an expected substring. The `@traceable` decorator logs each run to LangSmith so failures are inspectable after the fact.

In [ ]:
# ============================================================================
# INTEGRATION TEST SUITE: Real LLM Calls
# ============================================================================
class IntegrationTestSuite:
    """Integration tests with real LLM calls."""

    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @traceable(name="integration_test")
    def test_basic_qa(self) -> dict:
        """Test basic question answering."""
        test_cases = [
            {
                "question": "What is 2 + 2?",
                "expected_contains": ["4", "four"],
            },
            {
                "question": "What color is the sky on a clear day?",
                "expected_contains": ["blue"],
            },
        ]

        results = []
        for case in test_cases:
            response = self.llm.invoke(case["question"])
            content = response.content.lower()
            passed = any(exp.lower() in content for exp in case["expected_contains"])
            # "The answer is 4" or "2 + 2 equals four" or "That would be 4."
            results.append(
                {
                    "question": case["question"],
                    "response": response.content,
                    "passed": passed,
                }
            )

        return {
            "total": len(results),
            "passed": sum(1 for r in results if r["passed"]),
            "results": results,
        }

### ▶️ Demo: Run the Integration Tests

Calling this executes real LLM requests, so run it deliberately rather than on every save.

In [ ]:
# ============================================================================
# DEMO: Integration Test Results
# ============================================================================
def demo_integration_tests():
    """Run integration tests."""
    suite = IntegrationTestSuite()

    print("Integration Test Results:\n")
    results = suite.test_basic_qa()
    print(f"Passed: {results['passed']}/{results['total']}")

    for r in results["results"]:
        status = "✅" if r["passed"] else "❌"
        print(f"{status} {r['question']}")
        print(f"   Response: {r['response'][:50]}...")

---
## ⚖️ Part 3: LLM-as-Judge Evaluation Framework

Many quality dimensions — clarity, helpfulness, completeness — can't be checked with a simple string match. The **LLM-as-judge** pattern uses a second LLM call to grade the first one's output against a rubric, returning structured scores you can aggregate and track over time.

### Key Insight:
> LLM-as-judge scores are noisy and prompt-sensitive. Treat them as a directional signal for regression testing, not a certified ground truth.

### `LLMEvaluator`

Sends the question, response, and (optionally) a reference answer to an LLM and asks it to rate correctness, relevance, clarity, and completeness on a 1-10 scale, returned as JSON.

In [ ]:
# ============================================================================
# LLM EVALUATOR: Score a Response on Multiple Dimensions
# ============================================================================
class LLMEvaluator:
    """Use LLM to evaluate LLM outputs."""

    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    @traceable(name="evaluate_response")
    def evaluate(self, question: str, response: str, reference: str = None) -> dict:
        """Evaluate a response on multiple dimensions."""
        eval_prompt = ChatPromptTemplate.from_template(
            """Evaluate this response on a scale of 1-10 for each criterion.

Question: {question}
Response: {response}
{reference_section}

Rate each criterion (1-10):
1. Correctness: Is the information accurate?
2. Relevance: Does it answer the question?
3. Clarity: Is it easy to understand?
4. Completeness: Does it fully address the question?

Respond with ONLY a JSON object:
{{"correctness": X, "relevance": X, "clarity": X, "completeness": X, "overall": X}}"""
        )

        reference_section = ""
        if reference:
            reference_section = f"Reference answer: {reference}"

        import json

        response_obj = self.llm.invoke(
            eval_prompt.format(
                question=question,
                response=response,
                reference_section=reference_section,
            )
        )

        try:
            scores = json.loads(response_obj.content)
            return scores
        except json.JSONDecodeError:
            return {"error": "Failed to parse evaluation"}

### ▶️ Demo: Evaluate a Single Response

Grades one hand-written response against a reference answer to show the shape of the scores dict.

In [ ]:
# ============================================================================
# DEMO: LLM-as-Judge Evaluation
# ============================================================================
def demo_evaluation():
    """Demonstrate LLM evaluation."""
    evaluator = LLMEvaluator()

    # Test case
    question = "Explain what machine learning is in simple terms."
    response = "Machine learning is when computers learn from data instead of being explicitly programmed. It's like teaching a child by showing examples rather than giving them rules."
    reference = "Machine learning is a type of artificial intelligence where computers learn patterns from data to make predictions or decisions."

    print("LLM Evaluation Demo:\n")
    print(f"Question: {question}")
    print(f"Response: {response}")

    scores = evaluator.evaluate(question, response, reference)

    print("\nScores:")
    for metric, score in scores.items():
        print(f"  {metric}: {score}/10")

---
## 🔁 Part 4: Regression Testing

Once you have an evaluator, you can run it against a fixed suite of test cases every time the chain changes (new prompt, new model, new retriever) and watch for score drops. This is the same idea as software regression tests, just scored by an LLM instead of an `assert`.

### `RegressionTestRunner`

Runs a chain (any callable that takes a string and returns a string) against a list of `{"input", "expected"}` cases, scores each with `LLMEvaluator`, and marks a case as passed when its overall score clears a threshold.

In [ ]:
# ============================================================================
# REGRESSION TEST RUNNER: Score a Chain Against a Fixed Test Suite
# ============================================================================
class RegressionTestRunner:
    """Run regression tests against a test dataset."""

    def __init__(self, chain: Callable):
        self.chain = chain
        self.evaluator = LLMEvaluator()

    @traceable(name="regression_test")
    def run(self, test_cases: list[dict]) -> dict:
        """
        Run regression tests.

        test_cases: [{"input": ..., "expected": ...}, ...]
        """
        results = []
        total_score = 0

        for case in test_cases:
            # Get response from chain
            response = self.chain(case["input"])

            # Evaluate
            scores = self.evaluator.evaluate(
                question=case["input"],
                response=response,
                reference=case.get("expected"),
            )

            overall = scores.get("overall", 0)
            total_score += overall

            results.append(
                {
                    "input": case["input"],
                    "response": response,
                    "expected": case.get("expected"),
                    "scores": scores,
                    "passed": overall >= 7,  # Threshold
                }
            )

        return {
            "total": len(results),
            "passed": sum(1 for r in results if r["passed"]),
            "average_score": total_score / len(results) if results else 0,
            "results": results,
        }

### ▶️ Demo: Run a Regression Suite

Wraps a plain `ChatOpenAI` call as the chain-under-test and runs it through two quick cases.

In [ ]:
# ============================================================================
# DEMO: Regression Testing
# ============================================================================
def demo_regression_testing():
    """Demonstrate regression testing."""
    # Simple chain to test
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    def qa_chain(question: str) -> str:
        return llm.invoke(question).content

    # Test cases
    test_cases = [
        {
            "input": "What is Python?",
            "expected": "Python is a programming language known for its simplicity.",
        },
        {"input": "What is 10 * 5?", "expected": "50"},
    ]

    runner = RegressionTestRunner(qa_chain)

    print("\nRegression Test Results:\n")
    results = runner.run(test_cases)
    print(f"Passed: {results['passed']}/{results['total']}")
    print(f"Average Score: {results['average_score']:.1f}/10")

    for r in results["results"]:
        status = "✅" if r["passed"] else "❌"
        print(f"\n{status} {r['input']}")
        print(f"   Response: {r['response'][:50]}...")
        print(f"   Overall Score: {r['scores'].get('overall', 'N/A')}/10")

---
## 📈 Part 5: LangSmith Evaluation Datasets — Production Approach

The patterns above are ad-hoc: test cases live in Python lists and results only print to stdout. In production you want **persistent, versioned test suites** you can re-run, compare across experiments, and inspect in a dashboard. LangSmith's `Client` and `evaluate()` give you exactly that — a dataset stored server-side, plus experiment tracking so you can compare prompt v1 vs v2 side by side.

In [ ]:
# ============================================================================
# LANGSMITH SETUP: Client, Evaluation Runner, and Chain LLM
# ============================================================================
# ChatOpenAI, ChatPromptTemplate, traceable, and load_dotenv are already
# imported and loaded from the Setup cell above — only `evaluate` is new here.
from langsmith.evaluation import evaluate

client = Client()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ LangSmith client and chain LLM ready!")

### `create_eval_dataset`

Creates (or recreates) a named LangSmith dataset with a handful of question/answer pairs. Deleting an existing dataset with the same name before recreating it is convenient for a demo, but avoid that pattern once a dataset has real evaluation history you want to keep.

In [ ]:
# ============================================================================
# CREATE EVAL DATASET: Persist Test Cases to LangSmith
# ============================================================================
def create_eval_dataset():
    """Create a dataset with test cases in LangSmith."""
    dataset_name = "qa-eval-dataset"

    # Delete if exists (for demo purposes — don't do this in production)
    existing = list(client.list_datasets(dataset_name=dataset_name))
    if existing:
        client.delete_dataset(dataset_id=existing[0].id)

    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Q&A evaluation dataset for testing our chain",
    )

    # Add test examples — inputs and expected outputs
    examples = [
        {
            "inputs": {"question": "What is Python?"},
            "outputs": {
                "answer": "Python is a high-level programming language known for its readability and versatility."
            },
        },
        {"inputs": {"question": "What is 15 * 4?"}, "outputs": {"answer": "60"}},
        {
            "inputs": {"question": "What does HTML stand for?"},
            "outputs": {"answer": "HyperText Markup Language"},
        },
        {
            "inputs": {"question": "Name one benefit of exercise."},
            "outputs": {
                "answer": "Exercise improves cardiovascular health and reduces the risk of chronic diseases."
            },
        },
        {
            "inputs": {"question": "What is the capital of Japan?"},
            "outputs": {"answer": "Tokyo"},
        },
    ]

    for ex in examples:
        client.create_example(
            inputs=ex["inputs"], outputs=ex["outputs"], dataset_id=dataset.id
        )

    print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
    return dataset_name

### 🔗 The Chain Under Test

A minimal prompt-to-LLM chain (LCEL pipe) that `qa_target` below will wrap for evaluation.

In [ ]:
# ============================================================================
# CHAIN UNDER TEST: Prompt → LLM
# ============================================================================
prompt = ChatPromptTemplate.from_template("Answer this question concisely: {question}")
qa_chain = prompt | llm

### `qa_target`

LangSmith's `evaluate()` expects a target function with a fixed signature: accept a dict of inputs, return a dict of outputs. This adapts our LCEL chain to that contract.

In [ ]:
# ============================================================================
# QA TARGET: Adapter for LangSmith evaluate()
# ============================================================================
@traceable(name="qa_target")
def qa_target(inputs: dict) -> dict:
    """
    Target function for LangSmith evaluation.
    Must accept a dict (inputs) and return a dict (outputs).
    """
    response = qa_chain.invoke({"question": inputs["question"]})
    return {"answer": response.content}

### 🧑‍⚖️ Evaluators

LangSmith evaluators are plain functions of `(run, example)` that return a `{"key": ..., "score": ...}` dict. Below are two LLM-as-judge graders (`correctness`, `helpfulness`) and one cheap heuristic (`contains_answer`) — mixing both kinds is a common, cost-effective pattern.

In [ ]:
# ============================================================================
# EVALUATOR SETUP: Grader LLM
# ============================================================================
# Evaluator: checks correctness against reference using LLM-as-judge
eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

#### `correctness`

Grades whether the prediction matches the reference answer, returning a binary 1.0/0.0 score.

In [ ]:
# ============================================================================
# EVALUATOR: Correctness (LLM-as-Judge, Reference-Based)
# ============================================================================
def correctness(run, example) -> dict:
    """LLM-as-judge evaluator for correctness against reference answer."""
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    question = example.inputs.get("question", "")

    grade_prompt = ChatPromptTemplate.from_template(
        "You are a grader. Given a question, a submission, and a reference answer, "
        "determine if the submission is correct, accurate, and factual compared to "
        "the reference answer.\n\n"
        "Question: {question}\n"
        "Submission: {submission}\n"
        "Reference: {reference}\n\n"
        "Respond with ONLY 'Y' if correct or 'N' if incorrect."
    )

    result = eval_llm.invoke(
        grade_prompt.format(
            question=question, submission=prediction, reference=reference
        )
    )

    score = 1.0 if result.content.strip().upper() == "Y" else 0.0
    return {"key": "correctness", "score": score}

#### `helpfulness`

Grades whether the response is clear and helpful — no reference answer needed, which makes it usable even for open-ended questions.

In [ ]:
# ============================================================================
# EVALUATOR: Helpfulness (LLM-as-Judge, Reference-Free)
# ============================================================================
def helpfulness(run, example) -> dict:
    """LLM-as-judge evaluator for helpfulness (no reference needed)."""
    prediction = run.outputs.get("answer", "")
    question = example.inputs.get("question", "")

    grade_prompt = ChatPromptTemplate.from_template(
        "You are a grader. Given a question and a response, "
        "determine if the response is helpful, clear, and easy to understand.\n\n"
        "Question: {question}\n"
        "Response: {response}\n\n"
        "Respond with ONLY 'Y' if helpful or 'N' if not helpful."
    )

    result = eval_llm.invoke(
        grade_prompt.format(question=question, response=prediction)
    )

    score = 1.0 if result.content.strip().upper() == "Y" else 0.0
    return {"key": "helpfulness", "score": score}

#### `contains_answer`

A cheap, non-LLM evaluator: checks what fraction of the reference answer's key words show up in the prediction. Useful as a fast sanity check alongside the more expensive LLM-as-judge graders.

In [ ]:
# ============================================================================
# EVALUATOR: Contains Answer (Heuristic, No LLM Call)
# ============================================================================
# Custom evaluator: simple keyword check
def contains_answer(run, example) -> dict:
    """
    Custom evaluator — checks if the response contains
    key terms from the expected answer.
    """
    prediction = run.outputs.get("answer", "").lower()
    reference = example.outputs.get("answer", "").lower()

    # Extract key words from reference (words > 3 chars)
    key_words = [word for word in reference.split() if len(word) > 3]

    # Check if at least 50% of key words appear in prediction
    if not key_words:
        return {"key": "contains_answer", "score": 1.0}

    matches = sum(1 for word in key_words if word in prediction)
    score = matches / len(key_words)
    return {"key": "contains_answer", "score": score}

### `run_evaluation`

Runs `qa_target` against every example in the dataset, scores each with all three evaluators, and tags the run with an `experiment_prefix` so it shows up as a named experiment in the LangSmith dashboard.

In [ ]:
# ============================================================================
# RUN EVALUATION: Score qa_target Against the Dataset
# ============================================================================
def run_evaluation(dataset_name: str):
    """Run evaluation against the dataset."""
    print(f"\nRunning evaluation against '{dataset_name}'...\n")

    results = evaluate(
        qa_target,
        data=dataset_name,
        evaluators=[correctness, helpfulness, contains_answer],
        experiment_prefix="qa-chain-v1",  # Tags this run for comparison
        max_concurrency=2,
    )

    # Print summary
    print("\nEvaluation Results:")
    print("-" * 50)
    for result in results:
        question = result["run"].inputs.get("question", "N/A")
        answer = result["run"].outputs.get("answer", "N/A")
        print(f"\nQ: {question}")
        print(f"A: {answer[:80]}...")
        for eval_result in result["evaluation_results"]["results"]:
            print(f"  {eval_result.key}: {eval_result.score}")

    return results

### `run_comparison`

Builds a second chain with a more detailed prompt, runs it under a different `experiment_prefix` ("qa-chain-v2"), and points you at the LangSmith dashboard to compare the two experiments side by side — this is the payoff of using persistent, versioned datasets instead of ad-hoc test lists.

In [ ]:
# ============================================================================
# RUN COMPARISON: Second Experiment for Prompt A/B Comparison
# ============================================================================
def run_comparison(dataset_name: str):
    """
    Run a second experiment with a different config,
    then compare in LangSmith dashboard.
    """
    # New prompt — more detailed instructions
    detailed_prompt = ChatPromptTemplate.from_template(
        "Answer this question accurately and concisely. "
        "If it's a factual question, be precise. "
        "If it's a math question, show just the answer.\n\n"
        "Question: {question}"
    )
    v2_chain = detailed_prompt | llm

    @traceable(name="qa_target_v2")
    def qa_target_v2(inputs: dict) -> dict:
        response = v2_chain.invoke({"question": inputs["question"]})
        return {"answer": response.content}

    print("\nRunning v2 experiment for comparison...\n")

    results = evaluate(
        qa_target_v2,
        data=dataset_name,
        evaluators=[correctness, helpfulness, contains_answer],
        experiment_prefix="qa-chain-v2",  # Different prefix for comparison
        max_concurrency=2,
    )

    print("\nDone! Compare v1 vs v2 in LangSmith dashboard:")
    print("  → Go to your LangSmith project → Datasets → qa-eval-dataset")
    print("  → Click 'Compare Experiments' to see v1 vs v2 side by side")

    return results

---
## ▶️ Run the Demo

The original `__main__` guard, kept verbatim. Jupyter sets `__name__` to `"__main__"`, so this cell runs end to end: create the dataset, run the v1 experiment, then run the v2 experiment for comparison. This calls real LLM APIs and LangSmith, so run it deliberately.

In [ ]:
# ============================================================================
# RUN: End-to-End LangSmith Evaluation Demo
# ============================================================================
if __name__ == "__main__":
    print("=" * 60)
    print("LangSmith Evaluation Datasets Demo")
    print("=" * 60)

    # Step 1: Create dataset
    dataset_name = create_eval_dataset()

    # Step 2: Run first evaluation (v1)
    print("\n" + "=" * 60)
    print("Experiment 1: Basic prompt (v1)")
    print("=" * 60)
    run_evaluation(dataset_name)

    # Step 3: Run second evaluation (v2) for comparison
    print("\n" + "=" * 60)
    print("Experiment 2: Detailed prompt (v2)")
    print("=" * 60)
    run_comparison(dataset_name)

    print("\n" + "=" * 60)
    print("All experiments logged to LangSmith!")
    print("=" * 60)

---
## 📝 Summary

In this notebook, we built a full testing and evaluation toolkit for LLM applications, layered from fastest/cheapest to most production-realistic.

### 1. Testing Layers
- **Unit tests with mocks** (`QAChain`, `test_qa_chain_with_mock`, `test_qa_chain_handles_empty_response`): fast, deterministic, no API calls
- **Integration tests** (`IntegrationTestSuite`, `demo_integration_tests`): real LLM calls against a small, curated set of cases
- **LLM-as-judge evaluation** (`LLMEvaluator`, `demo_evaluation`): scores correctness, relevance, clarity, and completeness on a 1-10 scale
- **Regression testing** (`RegressionTestRunner`, `demo_regression_testing`): re-scores a chain against a fixed suite to catch quality drift over time

### 2. Production-Grade Evaluation with LangSmith
- **Persistent datasets** (`create_eval_dataset`): versioned test cases stored server-side instead of hardcoded Python lists
- **Target adapters** (`qa_target`): wrap an LCEL chain to match `evaluate()`'s `(inputs) -> outputs` contract
- **Mixed evaluators** (`correctness`, `helpfulness`, `contains_answer`): combine LLM-as-judge graders with cheap heuristics
- **Experiment comparison** (`run_evaluation`, `run_comparison`): tag runs with `experiment_prefix` to compare prompt v1 vs v2 side by side in the LangSmith dashboard

### Next Steps
- Apply this testing pyramid to your own LLM application: start with mocked unit tests for chain logic, add a handful of integration tests for real model behavior, then graduate ad-hoc evaluation scripts into a persistent LangSmith dataset once the chain is stable enough to track regressions against.
- Wire `test_qa_chain_with_mock`-style tests into CI (`pytest`) so prompt or chain changes can't silently break behavior.
- Revisit the `12_Production_and_Observability` phase's tracing and cost-monitoring notebooks to pair this evaluation layer with the observability needed to run these tests continuously in production.